# Data Extraction — Decision Tree Variables (Soybean)

Pulls the four agronomic variables straight from the raw CSV and computes their **occurrence counts and probabilities**. (The spreadsheet `decision_tree_soybeans.xlsx` is generated separately by `create_decision_tree.py`.)

- **Source:** `data/planting_summary_brazil.csv`
- **Bands:** D1 (planting date), C1 (soil texture) and D2 (seed population) follow the spreadsheet's definitions.
- **pH (C2) correction:** `Critical = ph_avg ≥ 6.15` (~4.30%) — fixes the prior 0% caused by thresholds clipped at the data boundary `[5.0, 6.8]`.
- Base/adjustment values (sc/ha) are Bayer slide constants; only the counts/probabilities come from the data.

In [1]:
import os, sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.getcwd())

PLANTING_CSV = "data/planting_summary_brazil.csv"
HARVEST_CSV  = "../Sprint 2/payoffMatrix/harvest_summary_brazil.csv"
OUTPUT_XLSX  = "decision_tree_soybeans.xlsx"
print("cwd:", os.getcwd())

cwd: /home/inteli/gitlab/g04/Sprint 3


## 1. Load planting data and filter soybeans

In [2]:
usecols = ["crop_name", "planting_date", "seed_actual_population_per_hectare",
           "soil_texture", "soil_ph_min", "soil_ph_max",
           "field_uuid", "concrete_growing_season_name"]
plant = pd.read_csv(PLANTING_CSV, usecols=usecols, low_memory=False)
print("Total planting rows:", f"{len(plant):,}")
soy = plant[plant["crop_name"].str.lower().str.contains("soy", na=False)].copy()
print("Soybean rows:", f"{len(soy):,}")

Total planting rows: 319,788
Soybean rows: 180,828


## 2. Cleaning (same rule as the spreadsheet)

Drop rows with missing **or** > 2,000,000 seeds/ha. This reproduces the spreadsheet's `N = 180,663`.

In [3]:
soy["seed"] = pd.to_numeric(soy["seed_actual_population_per_hectare"], errors="coerce")
before = len(soy)
clean = soy.dropna(subset=["seed"])
clean = clean[clean["seed"] <= 2_000_000].copy()
N = len(clean)
print(f"Removed (NaN or > 2,000,000 seeds/ha): {before - N}")
print(f"N (clean soybean records): {N:,}")

Removed (NaN or > 2,000,000 seeds/ha): 165
N (clean soybean records): 180,663


## 3. D1 — Planting Date

`Early = Sep/Oct`, `Normal = Nov`, `Late = every other month` (same band as the spreadsheet). Base yield is a slide constant.

In [4]:
clean["planting_month"] = pd.to_datetime(
    clean["planting_date"], format="%d-%m-%Y", errors="coerce").dt.month

def d1_window(m):
    if m in (9, 10): return "Early"
    if m == 11:      return "Normal"
    return "Late"

clean["D1"] = clean["planting_month"].apply(d1_window)
d1_counts = clean["D1"].value_counts()
d1_tbl = pd.DataFrame({
    "class":      ["Early", "Normal", "Late"],
    "window":     ["Sep-Oct", "Nov", "Dec-Mar"],
    "base_sc_ha": [64, 60, 52],
    "n":          [int(d1_counts[k]) for k in ["Early", "Normal", "Late"]],
})
d1_tbl["prob_%"] = (d1_tbl["n"] / N * 100).round(2)
d1_tbl

,class,window,base_sc_ha,n,prob_%
0,Early,Sep-Oct,64,103894,57.51
1,Normal,Nov,60,56386,31.21
2,Late,Dec-Mar,52,20383,11.28


## 4. C1 — Soil Texture

Portuguese texture labels mapped to the three slide classes (same grouping as the spreadsheet). Adjustment is a slide constant.

In [5]:
FAVORABLE   = {"argilosa", "muito argilosa", "média a argilosa", "argilosa a muito argilosa"}
CHALLENGING = {"arenosa"}

def c1_class(t):
    if t in FAVORABLE:   return "Favorable"
    if t in CHALLENGING: return "Challenging"
    return "Intermediary"

clean["C1"] = clean["soil_texture"].apply(c1_class)

known = FAVORABLE | CHALLENGING | {"franco-argilosa", "franca", "textura média", "média", "franca a franco-argilosa"}
unmapped = set(clean["soil_texture"].dropna().unique()) - known
print("Textures defaulting to Intermediary (unmapped):", unmapped or "none")

c1_counts = clean["C1"].value_counts()
c1_tbl = pd.DataFrame({
    "class":     ["Favorable", "Intermediary", "Challenging"],
    "adj_sc_ha": [4, 0, -6],
    "n":         [int(c1_counts[k]) for k in ["Favorable", "Intermediary", "Challenging"]],
})
c1_tbl["prob_%"] = (c1_tbl["n"] / N * 100).round(2)
c1_tbl

Textures defaulting to Intermediary (unmapped): none


,class,adj_sc_ha,n,prob_%
0,Favorable,4,86881,48.09
1,Intermediary,0,81509,45.12
2,Challenging,-6,12273,6.79


## 5. D2 — Seed Population

Real terciles by quantile (P33 / P66) of `seed_actual_population_per_hectare` — same as the spreadsheet. Adjustment is a slide constant.

In [6]:
p33 = clean["seed"].quantile(1/3)
p66 = clean["seed"].quantile(2/3)
print(f"P33 = {p33:,.0f}   P66 = {p66:,.0f} seeds/ha")

def d2_class(v):
    if v <= p33: return "Low"
    if v <= p66: return "Ideal"
    return "High"

clean["D2"] = clean["seed"].apply(d2_class)
d2_counts = clean["D2"].value_counts()
d2_tbl = pd.DataFrame({
    "class":     ["Low", "Ideal", "High"],
    "range":     [f"<= {p33:,.0f}", f"{p33:,.0f} - {p66:,.0f}", f"> {p66:,.0f}"],
    "adj_sc_ha": [-3, 2, -1],
    "n":         [int(d2_counts[k]) for k in ["Low", "Ideal", "High"]],
})
d2_tbl["prob_%"] = (d2_tbl["n"] / N * 100).round(4)
d2_tbl

P33 = 242,656   P66 = 290,345 seeds/ha


,class,range,adj_sc_ha,n,prob_%
0,Low,"<= 242,656",-3,60221,33.3333
1,Ideal,"242,656 - 290,345",2,60221,33.3333
2,High,"> 290,345",-1,60221,33.3333


## 6. C2 — Soil pH  *(methodological correction)*

`ph_avg = (soil_ph_min + soil_ph_max) / 2`. The pH data is **quantized** (coarse soil-survey grid) and clipped to `[5.0, 6.8]`, so the old `Critical = pH < 5.0 or > 6.8` matched **0** fields.

**Corrected bands (recalibrated to the observed distribution):**

| Class | pH range | Adj |
|---|---|---|
| Borderline | `≤ 5.4` | 0 |
| Adequate | `5.5 – 6.0` | +4 |
| **Critical** | `≥ 6.15` (alkaline extreme) | −6 |

Cutoffs (5.45 / 6.075) sit in the gaps of the discrete grid. A hard assert enforces `1% ≤ Critical ≤ 5%`.

In [7]:
clean["ph_avg"] = (pd.to_numeric(clean["soil_ph_min"], errors="coerce")
                   + pd.to_numeric(clean["soil_ph_max"], errors="coerce")) / 2

print("Distinct ph_avg values (quantized grid):")
print(clean["ph_avg"].round(2).value_counts().sort_index())

def c2_class(v):
    if pd.isna(v): return None
    if v < 5.45:   return "Borderline"   # pH <= 5.4
    if v < 6.075:  return "Adequate"     # pH 5.5 - 6.0
    return "Critical"                    # pH >= 6.15

clean["C2"] = clean["ph_avg"].apply(c2_class)
c2_counts = clean["C2"].value_counts(dropna=False)
c2_tbl = pd.DataFrame({
    "class":     ["Adequate", "Borderline", "Critical"],
    "pH_range":  ["5.5 - 6.0", "<= 5.4", ">= 6.15"],
    "adj_sc_ha": [4, 0, -6],
    "n":         [int(c2_counts.get(k, 0)) for k in ["Adequate", "Borderline", "Critical"]],
})
c2_tbl["prob_%"] = (c2_tbl["n"] / N * 100).round(3)

p_critical = c2_counts.get("Critical", 0) / N
print(f"\nP(Critical pH) = {p_critical*100:.3f}%")
assert 0.01 <= p_critical <= 0.05, "Critical pH share outside the required 1%-5% band!"
print("OK: Critical pH within the required 1%-5% band.")
c2_tbl

Distinct ph_avg values (quantized grid):
ph_avg
5.40    51038
5.50    53803
5.70    31196
6.00    36850
6.15     3312
6.30     4464
Name: count, dtype: int64

P(Critical pH) = 4.304%
OK: Critical pH within the required 1%-5% band.


,class,pH_range,adj_sc_ha,n,prob_%
0,Adequate,5.5 - 6.0,4,121849,67.445
1,Borderline,<= 5.4,0,51038,28.250
2,Critical,>= 6.15,-6,7776,4.304
